<a href="https://colab.research.google.com/github/vcasadei/Redes-Neurais-CESAR-School/blob/2026.01-A/5%20-%20Exemplos%20de%20Aplica%C3%A7%C3%A3o/5.7_gatos_e_cachorros_rede_densa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 5.7 — Gatos e cachorros: imagens reais com rede densa

O notebook 4.6 treinou uma rede densa no MNIST e passou de 98% de acurácia. Aqui usamos o **mesmo esqueleto** (mesma arquitetura, funções `train` e `test` equivalentes, agora com `nn.CrossEntropyLoss` e SGD com momentum, como nos outros exemplos de aplicação) num problema de imagem bem mais difícil: dizer se uma foto colorida mostra um **gato** ou um **cachorro**.

No MNIST, os dígitos estão centralizados, num fundo preto, todos do mesmo tamanho. Nas fotos de gatos e cachorros, o animal aparece em qualquer posição, tamanho, pose, cor e iluminação, com fundos variados. Uma rede densa vê cada pixel como uma entrada separada e não sabe quais pixels são vizinhos; o mesmo focinho, deslocado alguns pixels, é para ela uma entrada totalmente diferente. Vamos ver até onde ela chega e o que ajuda.

## Objetivos

- Carregar um conjunto de imagens reais (fotos JPEG de tamanhos diferentes), descartar arquivos corrompidos e reduzir as imagens para 32×32 pixels coloridos.
- Reaproveitar o pipeline do 4.6 com 3 canais de cor e 2 classes.
- Usar *data augmentation* (espelhamento horizontal aleatório) para reduzir o overfitting.
- Entender o limite das redes densas em imagens, que motiva as redes convolucionais.

## Importação das bibliotecas

In [ ]:
import os
import time
import urllib.request
import warnings
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from PIL import Image
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

warnings.filterwarnings('ignore', module='PIL')   # alguns JPEGs têm metadados malformados; os avisos não importam aqui
torch.manual_seed(1111)

device = "cuda" if torch.cuda.is_available() else "cpu"
print('device:', device)

## Dados

O conjunto [Microsoft Cats vs Dogs](https://www.kaggle.com/datasets/shaunthesheep/microsoft-catsvsdogs-dataset), do Kaggle, tem 25.000 fotos (12.500 de cada classe), em duas pastas: `PetImages/Cat` e `PetImages/Dog`. O zip tem cerca de 790 MB; no Colab, baixar e extrair leva um ou dois minutos. Se o download falhar, baixe o zip pela página do Kaggle e coloque-o em `data/`.

In [ ]:
os.makedirs("data", exist_ok=True)
url = "https://www.kaggle.com/api/v1/datasets/download/shaunthesheep/microsoft-catsvsdogs-dataset"
zip_path = "data/microsoft-catsvsdogs-dataset.zip"
root = "data/catsdogs/PetImages"
if not os.path.exists(zip_path):
    urllib.request.urlretrieve(url, zip_path)
if not os.path.exists(root):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall("data/catsdogs")
print({classe: len(os.listdir(os.path.join(root, classe))) for classe in ['Cat', 'Dog']})

### Carregar as imagens uma vez

No 4.6, o `DataLoader` lia cada imagem do disco e aplicava as transformações a cada época. Aqui, decodificar 25.000 JPEGs grandes a cada época deixaria o treino lento. Como as imagens reduzidas cabem na memória (25.000 × 32 × 32 × 3 bytes ≈ 77 MB), lemos e reduzimos todas **uma única vez** e guardamos num tensor.

Para cada arquivo:

- `Image.open(...).convert('RGB')`: abre a foto e garante 3 canais (vermelho, verde, azul); algumas fotos estão em tons de cinza ou têm canal de transparência;
- `.resize((32, 32))`: reduz para 32×32 pixels. É pouco, mas ainda dá para reconhecer o animal, e mantém a rede pequena (veja o exercício 2);
- `try`/`except`: alguns arquivos do conjunto estão corrompidos ou vazios; são descartados.

O rótulo é 0 para gato e 1 para cachorro. A célula leva cerca de 1 minuto.

In [ ]:
img_size = 32
classes = ['gato', 'cachorro']

t0 = time.time()
images, labels, descartadas = [], [], 0
for label, pasta in enumerate(['Cat', 'Dog']):
    for nome in sorted(os.listdir(os.path.join(root, pasta))):
        try:
            img = Image.open(os.path.join(root, pasta, nome))
            img.draft('RGB', (2 * img_size, 2 * img_size))   # acelera a leitura do JPEG já em tamanho reduzido
            img = img.convert('RGB').resize((img_size, img_size))
            images.append(np.asarray(img))
            labels.append(label)
        except Exception:
            descartadas += 1

images = torch.from_numpy(np.stack(images))   # (25000, 32, 32, 3), uint8 de 0 a 255
labels = torch.tensor(labels)
print(f'{len(images)} imagens carregadas, {descartadas} descartadas, em {time.time() - t0:.0f} s')
print('images:', tuple(images.shape), images.dtype, '| labels:', tuple(labels.shape))

In [ ]:
idx = torch.randperm(len(images))[:10]
fig, axs = plt.subplots(1, 10, figsize=(15, 2))
for ax, i in zip(axs, idx):
    ax.imshow(images[i])
    ax.set_title(classes[labels[i]])
    ax.axis('off')
plt.show()

### Treino, teste e normalização

- Sorteamos 80% das imagens para treino e 20% para teste.
- `permute(0, 3, 1, 2)` troca a ordem dos eixos de (N, altura, largura, canais), a ordem das imagens, para (N, canais, altura, largura), a ordem que o PyTorch usa (a mesma do `(1, 28, 28)` do MNIST).
- Dividir por 255 leva os pixels para [0, 1], como o `ToTensor()` do 4.6.
- Como o `Normalize((0.1307,), (0.3081,))` do 4.6, subtraímos a média e dividimos pelo desvio-padrão de cada canal, calculados no treino.

In [ ]:
perm = torch.randperm(len(images))
n_train = int(0.8 * len(images))
train_idx, test_idx = perm[:n_train], perm[n_train:]

x = images.permute(0, 3, 1, 2).float() / 255          # (N, 3, 32, 32), valores em [0, 1]
mean = x[train_idx].mean(dim=(0, 2, 3), keepdim=True)  # média de cada canal no treino: shape (1, 3, 1, 1)
std = x[train_idx].std(dim=(0, 2, 3), keepdim=True)
x = (x - mean) / std

print('média por canal (R, G, B):', mean.flatten().numpy().round(3))
print('desvio por canal (R, G, B):', std.flatten().numpy().round(3))

### Dataset com data augmentation

Um `Dataset` próprio, com os três métodos que já conhecemos: `__init__`, `__len__` e `__getitem__`. A novidade é o `transform`, aplicado a cada imagem **na hora em que ela é sorteada** para um mini-batch.

`transforms.RandomHorizontalFlip()` espelha a imagem na horizontal com probabilidade 50%. Um gato espelhado continua sendo um gato, e a rede passa a ver, a cada época, versões um pouco diferentes das mesmas fotos. Isso é **data augmentation**: aumentar a variedade do treino sem coletar dados novos. Na prática, funciona como regularização, assim como o dropout. O teste não usa augmentation: ele mede o desempenho nas imagens como elas são.

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, x, y, transform=None):
        self.x = x
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        img = self.x[i]
        if self.transform:
            img = self.transform(img)
        return img, self.y[i]

train_dataset = ImageDataset(x[train_idx], labels[train_idx], transform=transforms.RandomHorizontalFlip())
test_dataset = ImageDataset(x[test_idx], labels[test_idx])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000)

data, target = next(iter(train_loader))
print('treino:', len(train_dataset), 'imagens,', len(train_loader), 'mini-batches por época')
print('teste: ', len(test_dataset), 'imagens,', len(test_loader), 'lotes')
print('mini-batch: data', tuple(data.shape), ' target', tuple(target.shape))
print('fração de cachorros no treino:', labels[train_idx].float().mean().item())

As classes estão equilibradas (cerca de metade de cada). Isso dá a referência: chutando sempre a mesma classe, a acurácia seria de 50%.

## Modelo

É a rede do 4.6, com duas mudanças na entrada e na saída:

- `torch.flatten(x, 1)` achata cada imagem `(3, 32, 32)` num vetor de 3 × 32 × 32 = 3.072 números;
- `fc3` tem 2 saídas (gato, cachorro) em vez de 10.

O resto é igual: `fc1` (3072 → 512) e `fc2` (512 → 256) com ReLU e dropout depois de cada camada escondida (aqui `p = 0,3`, um pouco mais que no MNIST, porque o overfitting é maior).

A saída são os 2 **logits**, sem softmax: a `nn.CrossEntropyLoss`, que usamos no treino, já aplica o log-softmax internamente. Quando quisermos probabilidades, aplicamos `F.softmax` na saída.

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3 * img_size * img_size, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 2)
        self.dropout1 = nn.Dropout(0.3)
        self.dropout2 = nn.Dropout(0.3)

    def forward(self, x):          # x: (N, 3, 32, 32)
        x = torch.flatten(x, 1)    # (N, 3072)
        x = self.fc1(x)            # (N, 512)
        x = F.relu(x)
        x = self.dropout1(x)
        x = self.fc2(x)            # (N, 256)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc3(x)            # (N, 2): logits (sem softmax)
        return x

model = Net()
print(model)
print('parâmetros:', sum(p.numel() for p in model.parameters()))

## Treinamento

As funções `train` e `test` seguem as do 4.6: `model.train()`/`model.eval()`, `torch.no_grad()` no teste. A função de custo é a `nn.CrossEntropyLoss`, criada uma vez e passada como `criterion`; ela recebe os logits e os rótulos inteiros. `test` recebe também um nome, para podermos usá-la no conjunto de treino.

In [ ]:
def train(model, device, train_loader, optimizer, criterion, epoch, log_interval=150):
    model.train()                       # modo de treino: dropout ativo
    losses = []
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        if batch_idx % log_interval == 0:
            print('Época de treino: {} [{}/{} ({:.0f}%)]\tCusto: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))
    return losses

In [ ]:
def test(model, device, test_loader, criterion, nome='Teste'):
    model.eval()                        # modo de avaliação: dropout desligado
    test_loss = 0
    correct = 0
    with torch.no_grad():               # sem gradientes na avaliação
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item() * len(data)   # custo médio do lote × nº de amostras
            pred = output.argmax(dim=1, keepdim=True)  # índice do maior logit
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    accuracy = 100. * correct / len(test_loader.dataset)
    print('{}: custo médio: {:.4f}, acurácia: {}/{} ({:.2f}%)'.format(
        nome, test_loss, correct, len(test_loader.dataset), accuracy))
    return test_loss, accuracy

### Função de custo, otimizador e agendador

- **`nn.CrossEntropyLoss()`**: a entropia cruzada para classificação. Recebe os logits da rede e os rótulos inteiros (0 ou 1).
- **SGD com momentum** (`lr=0.01, momentum=0.9`): o gradiente descendente estocástico; o momentum suaviza os passos, levando em conta a direção dos passos anteriores.
- **StepLR** com `step_size=5, gamma=0.3`: a taxa de aprendizado cai para 30% a cada 5 épocas (0,01, 0,003, 0,0009).

São 15 épocas. Para acompanhar o overfitting, ao fim de cada época medimos também a acurácia no treino (sem augmentation e com o dropout desligado, nas mesmas condições do teste). Em CPU, o treino leva poucos minutos; numa GPU, menos de um minuto.

In [ ]:
model = Net().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
scheduler = StepLR(optimizer, step_size=5, gamma=0.3)

train_eval_loader = DataLoader(ImageDataset(x[train_idx], labels[train_idx]), batch_size=1000)   # treino sem augmentation

n_epochs = 15
losses, test_losses, test_accs, train_accs = [], [], [], []
t0 = time.time()
for epoch in range(1, n_epochs + 1):
    print(f'Época {epoch}: lr = {scheduler.get_last_lr()[0]:.5f}')
    losses += train(model, device, train_loader, optimizer, criterion, epoch)
    _, train_acc = test(model, device, train_eval_loader, criterion, nome='Treino')
    test_loss, test_acc = test(model, device, test_loader, criterion)
    train_accs.append(train_acc)
    test_losses.append(test_loss)
    test_accs.append(test_acc)
    scheduler.step()
    print()
print(f'tempo de treinamento: {time.time() - t0:.0f} s')

In [ ]:
n_batches = len(train_loader)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot([i / n_batches for i in range(len(losses))], losses, lw=0.5, label='treino (mini-batch)')
ax1.plot(range(1, n_epochs + 1), test_losses, 'o-', label='teste (fim da época)')
ax1.set_xlabel('épocas'); ax1.set_ylabel('custo (entropia cruzada)'); ax1.legend()
ax2.plot(range(1, n_epochs + 1), train_accs, 'o-', label='treino')
ax2.plot(range(1, n_epochs + 1), test_accs, 'o-', label='teste')
ax2.axhline(50, color='gray', ls='--', label='chute (50%)')
ax2.set_xlabel('épocas'); ax2.set_ylabel('acurácia (%)'); ax2.set_xticks(range(1, n_epochs + 1)); ax2.legend()
plt.show()

A rede acerta cerca de 2 em cada 3 fotos do teste: bem acima do chute, mas longe dos 98% do MNIST. A acurácia no treino fica acima da do teste, e a diferença cresce com as épocas: a rede passa a decorar fotos do treino. As quedas da taxa de aprendizado (épocas 6 e 11) dão saltos de melhora nas duas curvas.

## Gravação e leitura do modelo

Como no 4.6. Para usar o modelo depois, é preciso guardar também `mean` e `std`, para normalizar as imagens novas do mesmo jeito.

In [ ]:
torch.save(model.state_dict(), "gatos_cachorros_mlp.pt")

model2 = Net().to(device)
model2.load_state_dict(torch.load("gatos_cachorros_mlp.pt", map_location=device))
_ = test(model2, device, test_loader, criterion)

## Avaliação: exemplos do conjunto de teste

Predições para as 10 primeiras imagens do teste (verde: acerto, vermelho: erro) e, abaixo, outras 10 imagens em que a rede erra. Mostramos as imagens originais, sem normalização, e a probabilidade que a rede deu à classe prevista (softmax dos logits).

In [ ]:
model.eval()
with torch.no_grad():
    data, target = next(iter(test_loader))
    probs = F.softmax(model(data.to(device)), dim=1).cpu()   # logits → probabilidades
    preds = probs.argmax(dim=1)

originals = images[test_idx[:len(target)]]          # as mesmas imagens, sem normalização
wrong = (preds != target).nonzero().flatten()
wrong = wrong[wrong >= 10][:10]                    # erros fora das 10 primeiras imagens
fig, axs = plt.subplots(2, 10, figsize=(15, 3.8))
for ax, i in zip(axs[0], range(10)):
    ax.imshow(originals[i])
    cor = 'g' if preds[i] == target[i] else 'r'
    ax.set_title(f'{classes[preds[i]]} ({probs[i, preds[i]]:.0%})', fontsize=9, color=cor)
    ax.axis('off')
for ax, i in zip(axs[1], wrong):
    ax.imshow(originals[i])
    ax.set_title(f'{classes[preds[i]]} ({probs[i, preds[i]]:.0%})', fontsize=9, color='r')
    ax.axis('off')
plt.show()

Erros com probabilidade perto de 50% são casos em que a rede está em dúvida. Os erros com probabilidade alta costumam ser fotos em que a cor e o fundo enganam: a rede densa se apoia muito na distribuição de cores da imagem inteira, que é a informação que sobrevive ao achatamento, e pouco na forma do animal.

Acurácia por classe, no teste inteiro:

In [ ]:
all_preds, all_targets = [], []
with torch.no_grad():
    for data, target in test_loader:
        all_preds.append(model(data.to(device)).argmax(dim=1).cpu())
        all_targets.append(target)
all_preds, all_targets = torch.cat(all_preds), torch.cat(all_targets)
for c, nome in enumerate(classes):
    m = all_targets == c
    print(f'{nome:9s}: {(all_preds[m] == c).float().mean() * 100:.1f}% de acerto ({m.sum()} imagens)')

## Exercícios

1. Treine uma rede nova **sem** data augmentation (`transform=None` no `ImageDataset` do treino), com o mesmo otimizador, agendador e número de épocas, e anote as acurácias de treino e de teste da última época. Antes de executar, preveja o que acontece com cada uma. O augmentation ajudou?

In [ ]:
# TODO: escreva sua solução aqui
print(f'com augmentation: treino {train_accs[-1]:.1f}% | teste {test_accs[-1]:.1f}%')
print(f'sem augmentation: treino {train_acc_sem:.1f}% | teste {test_acc_sem:.1f}%')

*Escreva sua resposta aqui.*

2. Quantos parâmetros tem a camada `fc1` (entrada → 512) se as imagens forem 32×32 coloridas (como aqui), 64×64 coloridas e 128×128 em tons de cinza? Para comparar: a rede inteira deste notebook tem cerca de 1,7 milhão de parâmetros, e uma rede convolucional pequena, que passa de 80% neste problema com as mesmas imagens 32×32, tem algumas centenas de milhares. Por que aumentar a resolução da imagem não é uma boa saída para a rede densa?

In [ ]:
# TODO: escreva sua solução aqui


*Escreva sua resposta aqui.*

## Aprendizados

- O pipeline do 4.6 serve para qualquer conjunto de imagens: muda o carregamento dos dados, o número de canais na entrada e o número de classes na saída.
- Imagens reais exigem cuidados que o MNIST não exigia: arquivos corrompidos, tamanhos diferentes, conversão para RGB, e pré-processar uma vez quando o conjunto cabe na memória.
- Data augmentation (espelhar, por exemplo) aumenta a variedade do treino e reduz o overfitting, como o dropout.
- Uma rede densa trata a imagem como uma lista de pixels sem vizinhança: chega a cerca de 70% em gatos × cachorros, contra mais de 98% no MNIST. Aumentar a resolução só aumenta os parâmetros. Esse é o problema que as redes convolucionais resolvem.